# 04. Evaluation, Benchmarking & Stakeholder Validation

**Lead Responsible**: Weerasooriya W.H.M.S.P (`IT24104192`)  
**Module**: IT3091 – Machine Learning | Group 2026-AI-15  
**Track**: Industry Explorer Track  

---

## Objectives
1. Synthesize and evaluate top association rules using Support, Confidence, Lift (> 1.0), and Conviction.
2. Compare supervised classification algorithms (Logistic Regression vs. Decision Tree vs. Random Forest).
3. Cross-validate findings against the Sri Lankan POS reference dataset for staple FMCG goods (tea, dairy, biscuits).
4. Align insights with the independent supermarket owner's qualitative feedback and compile actionable store layout and restocking recommendations.

In [2]:
import sys
import time
from pathlib import Path

# Resolve project root dynamically
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats

# Scikit-learn Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    average_precision_score,
    roc_curve, 
    precision_recall_curve, 
    confusion_matrix
)

# Project modular imports
from src.utils import setup_plot_style, save_figure, export_summary_table

setup_plot_style()
print(f"[OK] Python Interpreter : {sys.executable}")
print(f"[OK] Project Root       : {PROJECT_ROOT}")

[OK] Python Interpreter : E:\SLIIT\Y3S1\IT3091 - Machine Learning\Assignments\SuperMarket\supermarket-basket-reorder-analytics\.venv\Scripts\python.exe
[OK] Project Root       : E:\SLIIT\Y3S1\IT3091 - Machine Learning\Assignments\SuperMarket\supermarket-basket-reorder-analytics


## 1. Ingestion of Upstream Analytical Artifacts
We ingest the 4 trained models, the held-out test set, CV scores, and association rules:
1. `models/model_1_logistic_regression.joblib` 
2. `models/model_2_decision_tree.joblib` 
3. `models/model_3_random_forest.joblib` 
4. `models/model_4_gradient_boosting.joblib` 
5. `data/processed/X_test.parquet` and `y_test.parquet` 
6. `reports/tables/reorder_cv_fold_scores.csv` 
7. `reports/tables/top_association_rules.csv` 

### Load All 4 Models & Test Partitions

In [3]:
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

# 1. Load Test Partitions
X_test = pd.read_parquet(PROCESSED_DIR / "X_test.parquet")
y_test = pd.read_parquet(PROCESSED_DIR / "y_test.parquet")["reordered"].values


In [4]:
# 2. Load Trained Candidate Models
models = {
    "Logistic Regression": joblib.load(MODELS_DIR / "model_1_logistic_regression.joblib"),
    "Decision Tree": joblib.load(MODELS_DIR / "model_2_decision_tree.joblib"),
    "Random Forest": joblib.load(MODELS_DIR / "model_3_random_forest.joblib"),
    "HistGradientBoosting": joblib.load(MODELS_DIR / "model_4_gradient_boosting.joblib")
}

In [5]:
# 3. Load CV Scores and Rules
cv_scores_df = pd.read_csv(TABLES_DIR / "reorder_cv_fold_scores.csv")
rules_df = pd.read_csv(TABLES_DIR / "top_association_rules.csv")

print("=" * 65)
print("UPSTREAM ARTIFACTS INGESTION SUMMARY")
print("=" * 65)
print(f"Test Partition Loaded       : {len(X_test):,} samples")
print(f"Candidate Models Ingested   : {len(models)} models")
print(f"CV Fold Records Ingested    : {len(cv_scores_df)} records")
print(f"Association Rules Ingested  : {len(rules_df)} rules")
print("=" * 65)

UPSTREAM ARTIFACTS INGESTION SUMMARY
Test Partition Loaded       : 30,000 samples
Candidate Models Ingested   : 4 models
CV Fold Records Ingested    : 43 records
Association Rules Ingested  : 4047 rules


## 2. Independent Test Set Evaluation Across All 4 Models
We generate predictions and probability scores for each candidate model on the unseen test set, calculating:
* **Accuracy & F1-Score (Macro / Weighted)**
* **Precision & Recall** (Critical for retail inventory trade-offs)
* **ROC-AUC & PR-AUC** (Average Precision)

### Compute Comprehensive Evaluation Metrics

In [6]:
benchmark_results = []
predictions_dict = {}

for name, model in models.items():
    t0 = time.time()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    latency_ms = ((time.time() - t0) / len(X_test)) * 1000
    
    predictions_dict[name] = {"pred": y_pred, "proba": y_proba}
    
    benchmark_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1-Score (Macro)": round(f1_score(y_test, y_pred, average="macro"), 4),
        "F1-Score (Weighted)": round(f1_score(y_test, y_pred, average="weighted"), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4),
        "PR-AUC": round(average_precision_score(y_test, y_proba), 4),
        "Inference_Latency_ms": round(latency_ms, 4)
    })

df_benchmarks = pd.DataFrame(benchmark_results).sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)
export_summary_table(df_benchmarks, "reorder_model_benchmarks.csv")

print("=" * 70)
print("INDEPENDENT TEST SET BENCHMARK PERFORMANCE TABLE")
print("=" * 70)
display(df_benchmarks)

[Exported Table]  -> reports\tables\reorder_model_benchmarks.csv
INDEPENDENT TEST SET BENCHMARK PERFORMANCE TABLE


,Model,Accuracy,Precision,Recall,F1-Score (Macro),F1-Score (Weighted),ROC-AUC,PR-AUC,Inference_Latency_ms
0,Random Forest,0.7563,0.8025,0.7780,0.7498,0.7570,0.8296,0.8531,0.0099
1,HistGradientBoosting,0.7604,0.7752,0.8360,0.7477,0.7578,0.8268,0.8494,0.0052
2,Logistic Regression,0.7411,0.7863,0.7699,0.7337,0.7416,0.8161,0.8491,0.0077
3,Decision Tree,0.7394,0.7909,0.7582,0.7330,0.7404,0.8095,0.8319,0.0005


## 3. Statistical Significance Hypothesis Testing (Paired t-Test)
To satisfy rigorous academic standards, we conduct **Paired Student’s t-Tests** on the 5-fold cross-validation scores between our winning ensemble (Random Forest) and candidate alternatives.  
* **$H_0$ (Null Hypothesis)**: There is no significant difference in mean performance between models.  
* **$H_1$ (Alternative Hypothesis)**: The ensemble model significantly outperforms the candidate model ($\alpha = 0.05$).

### Execute Paired t-Tests Across CV Folds

In [7]:
rf_cv = cv_scores_df[cv_scores_df["Model"] == "Random Forest"]["ROC_AUC"].values
lr_cv = cv_scores_df[cv_scores_df["Model"] == "Logistic Regression"]["ROC_AUC"].values
dt_cv = cv_scores_df[cv_scores_df["Model"] == "Decision Tree"]["ROC_AUC"].values

# Paired t-test: RF vs Baseline LR
t_stat_lr, p_val_lr = stats.ttest_rel(rf_cv, lr_cv)
# Paired t-test: RF vs Decision Tree
t_stat_dt, p_val_dt = stats.ttest_rel(rf_cv, dt_cv)

print("=" * 65)
print("STATISTICAL SIGNIFICANCE HYPOTHESIS TESTING (PAIRED t-TEST)")
print("=" * 65)
print(f"Random Forest vs. Logistic Regression : t-statistic = {t_stat_lr:.3f}, p-value = {p_val_lr:.4e}")
print(f"  -> Result: {'Statistically Significant (p < 0.05) - Reject H0' if p_val_lr < 0.05 else 'Not Significant'}")
print(f"Random Forest vs. Decision Tree       : t-statistic = {t_stat_dt:.3f}, p-value = {p_val_dt:.4e}")
print(f"  -> Result: {'Statistically Significant (p < 0.05) - Reject H0' if p_val_dt < 0.05 else 'Not Significant'}")
print("=" * 65)

STATISTICAL SIGNIFICANCE HYPOTHESIS TESTING (PAIRED t-TEST)
Random Forest vs. Logistic Regression : t-statistic = 11.492, p-value = 1.1121e-06
  -> Result: Statistically Significant (p < 0.05) - Reject H0
Random Forest vs. Decision Tree       : t-statistic = 12.053, p-value = 7.4158e-07
  -> Result: Statistically Significant (p < 0.05) - Reject H0
